In [13]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import pandas as pd

def extract_links(url):
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url

    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return pd.DataFrame(columns=['link_text', 'url', 'link_type', 'file_extension'])

    soup = BeautifulSoup(response.text, 'html.parser')

    base_url = url

    link_texts = []
    link_urls = []
    link_types = []
    file_extensions = []

    for a_tag in soup.find_all('a', href=True):
        href = a_tag.get('href')
        if not href or href.startswith(('javascript:', '#')):
            continue

        absolute_url = urljoin(base_url, href)
        link_text = a_tag.get_text(strip=True) or "[No Text]"

        parsed_url = urlparse(absolute_url)
        path = parsed_url.path.lower()
        file_extension = path.split('.')[-1] if '.' in path else ''

        if file_extension in ['pdf', 'doc', 'docx', 'xls', 'xlsx', 'ppt', 'pptx', 'txt', 'csv']:
            link_type = "Download"
        else:
            link_type = "Hyperlink"

        link_texts.append(link_text)
        link_urls.append(absolute_url)
        link_types.append(link_type)
        file_extensions.append(file_extension if file_extension else "None")

    df = pd.DataFrame({
        'link_text': link_texts,
        'url': link_urls,
        'link_type': link_types,
        'file_extension': file_extensions
    })

    df = df.drop_duplicates(subset=['url'])

    return df


In [14]:
url = "https://www.cok.agh.edu.pl/regulamin-studiow-agh/regulamin-studiow-agh-tekst"

print(f"Extracting links from {url}...")
links_df = extract_links(url)

if len(links_df) == 0:
    print("No links found or could not access the website.")
else:
    print(f"Found {len(links_df)} unique links.")

    print("\nFirst 5 links:")
    print(links_df.head())

    output_file = "extracted_links.csv"
    links_df.to_csv(output_file, index=False)
    print(f"\nAll links have been saved to {output_file}")

    print("\nLink type distribution:")
    print(links_df['link_type'].value_counts())

Extracting links from https://www.cok.agh.edu.pl/regulamin-studiow-agh/regulamin-studiow-agh-tekst...
Found 24 unique links.

First 5 links:
                     link_text                                     url  \
0                    [No Text]                  https://www.agh.edu.pl   
1  Centrum Obsługi Kształcenia             https://www.cok.agh.edu.pl/   
2                        O nas        https://www.cok.agh.edu.pl/o-nas   
4                   Zespół COK       https://www.cok.agh.edu.pl/zespol   
5                  Akty prawne  https://www.cok.agh.edu.pl/akty-prawne   

   link_type file_extension  
0  Hyperlink           None  
1  Hyperlink           None  
2  Hyperlink           None  
4  Hyperlink           None  
5  Hyperlink           None  

All links have been saved to extracted_links.csv

Link type distribution:
link_type
Hyperlink    21
Download      3
Name: count, dtype: int64


In [15]:
links_df

,link_text,url,link_type,file_extension
0,[No Text],https://www.agh.edu.pl,Hyperlink,None
1,Centrum Obsługi Kształcenia,https://www.cok.agh.edu.pl/,Hyperlink,None
2,O nas,https://www.cok.agh.edu.pl/o-nas,Hyperlink,None
4,Zespół COK,https://www.cok.agh.edu.pl/zespol,Hyperlink,None
5,Akty prawne,https://www.cok.agh.edu.pl/akty-prawne,Hyperlink,None
6,Systemy i serwisy w AGH,https://www.cok.agh.edu.pl/systemy-i-serwisy-w...,Hyperlink,None
7,Niezbędnik studenta,https://www.cok.agh.edu.pl/niezbednik,Hyperlink,None
9,Organizacja roku akademickiego,https://www.cok.agh.edu.pl/niezbednik/organiza...,Hyperlink,None
10,Regulamin studiów AGH,https://www.cok.agh.edu.pl/regulamin-studiow-agh,Hyperlink,None
11,Opłaty za studia,https://www.cok.agh.edu.pl/oplaty,Hyperlink,None
